# Multi-Layer Networks and Forward Pass Notebook

> Hands-on Build It and Exercises.

## Build It

Pure Python. No numpy. Every matrix operation written from scratch.

### Step 1: Sigmoid Activation

In [ ]:
```python

import math

def sigmoid(x):

    x = max(-500.0, min(500.0, x))

    return 1.0 / (1.0 + math.exp(-x))

In [ ]:
```

The clamp to [-500, 500] prevents overflow. `math.exp(500)` is large but finite. `math.exp(1000)` is infinity.

### Step 2: Layer Class

The most important operation in all of deep learning is matrix multiplication. Every layer, every attention head, every forward pass -- it's matmuls all the way down. A linear layer takes an input vector, multiplies it by a weight matrix, and adds a bias vector: y = Wx + b. That single equation is 90% of the compute in a neural network.

A layer holds a weight matrix and a bias vector. Its forward method takes an input vector and returns the activated output.

In [ ]:
```python

class Layer:

    def __init__(self, n_inputs, n_neurons, weights=None, biases=None):

        if weights is not None:

            self.weights = weights

        else:

            import random

            self.weights = [

                [random.uniform(-1, 1) for _ in range(n_inputs)]

                for _ in range(n_neurons)

            ]

        if biases is not None:

            self.biases = biases

        else:

            self.biases = [0.0] * n_neurons

    def forward(self, inputs):

        self.last_input = inputs

        self.last_output = []

        for neuron_idx in range(len(self.weights)):

            z = sum(

                w * x for w, x in zip(self.weights[neuron_idx], inputs)

            )

            z += self.biases[neuron_idx]

            self.last_output.append(sigmoid(z))

        return self.last_output

In [ ]:
```

The weight matrix has shape (n_neurons, n_inputs). Each row is one neuron's weights across all inputs. The forward method loops through neurons, computes the weighted sum plus bias, applies sigmoid, and collects the results.

### Step 3: Network Class

A network is a list of layers. The forward pass chains them: output of layer k feeds into layer k+1.

In [ ]:
```python

class Network:

    def __init__(self, layers):

        self.layers = layers

    def forward(self, inputs):

        current = inputs

        for layer in self.layers:

            current = layer.forward(current)

        return current

In [ ]:
```

That is the entire forward pass. Four lines of logic. Data goes in, flows through every layer, comes out the other side.

### Step 4: XOR with Hand-Tuned Weights

In Lesson 01, we solved XOR by combining OR, NAND, and AND perceptrons. Now do the same thing with our Layer and Network classes. The 2-2-1 architecture: two inputs, two hidden neurons, one output.

In [ ]:
```python

hidden = Layer(

    n_inputs=2,

    n_neurons=2,

    weights=[[20.0, 20.0], [-20.0, -20.0]],

    biases=[-10.0, 30.0],

)

output = Layer(

    n_inputs=2,

    n_neurons=1,

    weights=[[20.0, 20.0]],

    biases=[-30.0],

)

xor_net = Network([hidden, output])

xor_data = [

    ([0, 0], 0),

    ([0, 1], 1),

    ([1, 0], 1),

    ([1, 1], 0),

]

for inputs, expected in xor_data:

    result = xor_net.forward(inputs)

    predicted = 1 if result[0] >= 0.5 else 0

    print(f"  {inputs} -> {result[0]:.6f} (rounded: {predicted}, expected: {expected})")

In [ ]:
```

The large weights (20, -20) make sigmoid act like a step function. The first hidden neuron approximates OR. The second approximates NAND. The output neuron combines them into AND, which is XOR.

### Step 5: Circle Classification

A harder problem: classify 2D points as inside or outside a circle of radius 0.5 centered at the origin. This requires a curved decision boundary -- impossible for a single perceptron.

In [ ]:
```python

import random

import math

random.seed(42)

data = []

for _ in range(200):

    x = random.uniform(-1, 1)

    y = random.uniform(-1, 1)

    label = 1 if (x * x + y * y) < 0.25 else 0

    data.append(([x, y], label))

circle_net = Network([

    Layer(n_inputs=2, n_neurons=8),

    Layer(n_inputs=8, n_neurons=1),

])

In [ ]:
```

With random weights, the network will not classify well. But the forward pass still runs. This is the point -- the forward pass is just computation. Learning the right weights is backpropagation, coming in Lesson 03.

In [ ]:
```python

correct = 0

for inputs, expected in data:

    result = circle_net.forward(inputs)

    predicted = 1 if result[0] >= 0.5 else 0

    if predicted == expected:

        correct += 1

print(f"Accuracy with random weights: {correct}/{len(data)} ({100*correct/len(data):.1f}%)")

In [ ]:
```

Random weights give poor accuracy -- often worse than guessing the majority class. After training (Lesson 03), this same architecture with 8 hidden neurons will draw a curved boundary that separates inside from outside.

## Exercises

In [ ]:
1. Build a 2-4-2-1 network (two hidden layers) and run the forward pass on XOR data with random weights. Print the intermediate hidden layer outputs to see how the representation transforms at each layer.

2. Change the hidden layer size in the circle classifier from 8 to 2, then to 32. Run the forward pass with random weights each time. Does the number of hidden neurons change the output range or distribution? Why?

3. Implement a `count_parameters` method on the Network class that returns the total number of trainable weights and biases. Test it on a 784-256-128-10 network (the classic MNIST architecture). How many parameters does it have?

4. Build a forward pass for a 3-4-4-2 network. Feed it RGB color values (normalized to 0-1) and observe the two outputs. This is the architecture for a simple color classifier with two classes.

5. Replace sigmoid with a "leaky step" function: return 0.01 * z if z < 0, else 1.0. Run the forward pass on XOR with the same hand-tuned weights from Step 4. Does it still work? Why is the smooth sigmoid preferred over hard cutoffs?